In [1]:
from pathlib import Path
from typing import Dict, List, Tuple

import arviz as az
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

import cmdstanpy
cmdstanpy.rebuild_cmdstan()
from cmdstanpy import CmdStanModel


import kagglehub
from pathlib import Path
path = kagglehub.dataset_download("annavictoria/speed-dating-experiment")

print("Path to dataset files:", path)

import os

files = os.listdir(path)
print(files)


/Users/nidhipad/Dropbox/Mac/Downloads/Cognitive-Modeling-HW4/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Done:  (00:59) | ██████████ | --- CmdStan v2.38.0 built ---         


Path to dataset files: /Users/nidhipad/.cache/kagglehub/datasets/annavictoria/speed-dating-experiment/versions/1
['Speed Dating Data Key.doc', 'Speed Dating Data.csv']


In [2]:
# define path to csv, stan file, chosen predictor attributes from dataset, and target column from dataset

In [3]:
csv_path = Path(path)/ "Speed Dating Data.csv"
stan_file = Path("mpt_p4.stan")
predictor_cols = ["attr", "sinc", "intel"]
target_col = "dec"

In [4]:
# function to read csv, extract predictor columns + target columns, and clean extracted data
def load_and_clean_data(
    csv_path: Path,
    predictors: List[str],
    target: str,
) -> pd.DataFrame:

    df = pd.read_csv(csv_path, encoding="latin1")
    needed_cols = predictors + [target]
    df = df[needed_cols].dropna().copy()

    for col in predictors + [target]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna().copy()
    df[target] = df[target].astype(int)

    valid_targets = {0, 1}
    df = df[df[target].isin(valid_targets)].copy()

    return df

In [5]:
# function to compute the mean & std from the train set, then standardize both the train and test set using the computed values
def standardize_train_test(
    x_train: pd.DataFrame,
    x_test: pd.DataFrame,
) -> Tuple[np.ndarray, np.ndarray, pd.Series, pd.Series]:

    train_means = x_train.mean(axis=0)
    train_stds = x_train.std(axis=0, ddof=0)

    train_stds = train_stds.replace(0, 1.0)

    x_train_std = ((x_train - train_means) / train_stds).to_numpy()
    x_test_std = ((x_test - train_means) / train_stds).to_numpy()

    return x_train_std, x_test_std, train_means, train_stds


In [7]:
# function to package training data into Stan format
def build_stan_data(x_train: np.ndarray, y_train: np.ndarray, x_test:np.ndarray) -> Dict[str, object]:
    return {
        "N": x_train.shape[0],
        "K": x_train.shape[1],
        "X": x_train,
        "y": y_train.astype(int),
        "M": x_test.shape[0],
        "X_test": x_test,
    }

In [8]:
# function which compiles Stan model & samples from the posterior using MCMC
def fit_bayesian_logistic_regression(
    stan_file: Path,
    stan_data: Dict[str, object],
):
    model = CmdStanModel(stan_file=str(stan_file))
    fit = model.sample(
        data=stan_data,
        chains=4,
        parallel_chains=4,
        iter_warmup=1000,
        iter_sampling=1000,
        seed=42,
        refresh=200,
    )
    return fit

In [9]:
# function to extract posterior summary for the coefficients
def extract_beta_summary(fit, predictor_names):
    summary = fit.summary()
    beta_rows = [idx for idx in summary.index if idx.startswith("beta[")]

    if len(beta_rows) != len(predictor_names):
        raise ValueError(
            f"Mismatch: Stan produced {len(beta_rows)} beta coefficients, "
            f"but predictor_names has {len(predictor_names)} entries.\n"
            f"beta rows: {beta_rows}\n"
            f"predictor_names: {predictor_names}"
        )

    rows = []
    for row_name, predictor_name in zip(beta_rows, predictor_names):
        row = {
            "parameter": predictor_name,
        }

        for col in summary.columns:
            row[col] = summary.loc[row_name, col]

        rows.append(row)

    return pd.DataFrame(rows)

In [10]:
# function which extracts and averages the posterior probabilities computed in generated quantities block of Stan file
def posterior_mean_probabilities(fit) -> np.ndarray:
    ptest_draws=fit.stan_variable("p_test")
    return ptest_draws.mean(axis=0)

In [11]:
# function to compute Brier score
def brier_score(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    return float(np.mean((y_prob - y_true) ** 2))

In [12]:
# function to compute metric for each MCMC draw
def posterior_metric_distributions(fit, y_test: np.ndarray,) -> Tuple[np.ndarray, np.ndarray]:
    p_test_draws = fit.stan_variable("p_test")

    draw_brier = np.mean((p_test_draws - y_test[None, :]) ** 2, axis=1)

    draw_preds = (p_test_draws >= 0.5).astype(int)
    draw_accuracy = np.mean(draw_preds == y_test[None, :], axis=1)

    return draw_brier, draw_accuracy

In [13]:
# function to check whether the model converged
def convergence_check(summary_df: pd.DataFrame) -> bool:
    rhat_ok = (summary_df["r_hat"] < 1.01).all()
    ess_bulk_ok = (summary_df["ess_bulk"] > 400).all()
    ess_tail_ok = (summary_df["ess_tail"] > 400).all()
    return bool(rhat_ok and ess_bulk_ok and ess_tail_ok)

In [14]:
# function to identify the strongest predictor by taking the coefficient summary and selecting the predictor with the largest absolute posterior mean
def strongest_predictor(summary_df: pd.DataFrame) -> str:
    """
    use largest absolute posterior mean among the 3 predictors to determine strongest predictor
    """
    coef_df = summary_df[summary_df["parameter"] != "alpha"].copy()
    coef_df["abs_mean"] = coef_df["mean"].abs()
    best_row = coef_df.sort_values("abs_mean", ascending=False).iloc[0]
    return str(best_row["parameter"])

In [17]:
# main pipeline
def main() -> None:

    df = load_and_clean_data(
        csv_path=csv_path,
        predictors=predictor_cols,
        target=target_col,
    )

    x = df[predictor_cols]
    y = df[target_col].to_numpy()

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y,
    )

    x_train_std, x_test_std, train_means, train_stds = standardize_train_test(
        x_train=x_train,
        x_test=x_test,
    )

    stan_data = build_stan_data(x_train=x_train_std, y_train=y_train, x_test=x_test_std)
    fit = fit_bayesian_logistic_regression(
        stan_file=stan_file,
        stan_data=stan_data,
    )

    coef_summary = extract_beta_summary(
        fit=fit,
        predictor_names=predictor_cols,
    )

    coef_summary = coef_summary.rename(
        columns={
            "Mean": "mean",
            "StdDev": "sd",
            "R_hat": "r_hat",
            "ESS_bulk": "ess_bulk",
            "ESS_tail": "ess_tail",
        }
    )

    y_prob = posterior_mean_probabilities(fit=fit)
    y_pred = (y_prob >= 0.5).astype(int)
    test_brier=brier_score(y_test, y_prob)

    test_accuracy = accuracy_score(y_test, y_pred)
    test_auc = roc_auc_score(y_test, y_prob)

    draw_brier, draw_accuracy = posterior_metric_distributions(fit=fit, y_test=y_test)

    brier_ci_lower, brier_ci_upper=np.quantile(draw_brier, q=[0.025,0.975])
    acc_ci_lower, acc_ci_upper=np.quantile(draw_accuracy, q=[0.025,0.975])

    converge = convergence_check(coef_summary)
    best_predictor = strongest_predictor(coef_summary)

    print("\nSelected predictors:")
    print(predictor_cols)

    print("\nTraining standardization means:")
    print(train_means)

    print("\nTraining standardization standard deviations:")
    print(train_stds)

    print("\nPosterior summary:")
    print(coef_summary.to_string(index=False))

    print("\nConvergence result:")
    print(f"Model converged: {converge}")

    print("\nStrongest predictor of a 'Yes' decision:")
    print(best_predictor)

    print("\nTest-set performance:")
    print(f"Brier score: {test_brier:.4f}")
    print(f"Accuracy:    {test_accuracy:.4f}")
    print(f"ROC AUC:     {test_auc:.4f}")

    print("\nPosterior distribution over test metrics:")
    print(
        f"Brier 95% credible interval: "
        f"[{brier_ci_lower:.4f}, {brier_ci_upper:.4f}]"
    )
    print(
        f"Accuracy 95% credible interval: "
        f"[{acc_ci_lower:.4f}, {acc_ci_upper:.4f}]"
    )


In [18]:
if __name__ == "__main__":
    main()

20:50:51 - cmdstanpy - INFO - compiling stan file /Users/nidhipad/Dropbox/Mac/Downloads/Cognitive-Modeling-HW4/mpt_p4.stan to exe file /Users/nidhipad/Dropbox/Mac/Downloads/Cognitive-Modeling-HW4/mpt_p4
20:50:56 - cmdstanpy - INFO - compiled model executable: /Users/nidhipad/Dropbox/Mac/Downloads/Cognitive-Modeling-HW4/mpt_p4
20:50:57 - cmdstanpy - INFO - CmdStan start processing
chain 2:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]


chain 1:  10%|█         | 200/2000 [00:00<00:02, 616.35it/s, (Warmup)]


chain 2:  10%|█         | 200/2000 [00:00<00:03, 578.61it/s, (Warmup)]

chain 1:  20%|██        | 400/2000 [00:00<00:02, 721.02it/s, (Warmup)]


chain 2:  20%|██        | 400/2000 [00:00<00:02, 695.47it/s, (Warmup)]

chain 1:  30%|███       | 600/2000 [00:00<00:01, 745.60it/s, (Warmup)]


chain 2:  30%|███       | 600/2000 [00:00<00:01, 739.80it/s, (Warmup)]

chain 1:  40%|████      | 800/2000 [00:01<00:01, 824.93it/s


20:51:04 - cmdstanpy - INFO - CmdStan done processing.




Selected predictors:
['attr', 'sinc', 'intel']

Training standardization means:
attr     6.220664
sinc     7.206566
intel    7.386697
dtype: float64

Training standardization standard deviations:
attr     1.945831
sinc     1.736823
intel    1.546757
dtype: float64

Posterior summary:
parameter     mean     MCSE       sd      MAD        5%      50%      95%  ess_bulk  ess_tail  ESS_bulk/s   r_hat
     attr 1.291570 0.000673 0.040098 0.039651  1.224320 1.291720 1.357670   3561.67   2959.80     154.687 1.00148
     sinc 0.010495 0.000731 0.041808 0.041268 -0.058114 0.010452 0.081381   3338.85   2672.02     145.010 1.00013
    intel 0.111009 0.000757 0.041066 0.042314  0.044294 0.111489 0.178019   3004.42   2538.83     130.485 1.00043

Convergence result:
Model converged: True

Strongest predictor of a 'Yes' decision:
attr

Test-set performance:
Brier score: 0.1839
Accuracy:    0.7288
ROC AUC:     0.7912

Posterior distribution over test metrics:
Brier 95% credible interval: [0.1834, 0.1